In [ ]:
# ┌────────────────────────────┐
# │        Start Pipeline      │
# └──────────────┬─────────────┘
#                │
#                ▼
# ┌────────────────────────────────────────────────────────┐
# │ Input Params                                           │
# │ process_type = Inventory / Operations                  │
# │ namespace = ABC / XYZ / etc                            │
# │ date_status_update_flag (True/False)                   │
# │ status_update_flag (True/False)                        │
# │ status = Started / Completed / Failed                  │
# └──────────────┬─────────────────────────────────────────┘
#                │
#                ▼
# ┌────────────────────────────────────────────────────────┐
# │ Validate Inputs                                        │
# │ - process_type must be allowed                         │
# │ - namespace must not be null                           │
# │ - at least one update flag must be True                │
# │ - date must be datetime.date                           │
# └──────────────┬─────────────────────────────────────────┘
#                │
#                ▼
# ┌───────────────────────────────────────────────────────┐
# │ Check if Delta table exists at watermark_path         │
# │ DeltaTable.isDeltaTable(spark, path)                  │
# └──────────────┬────────────────────────────────────────┘
#        ┌───────┴─────────┐
#        │                 │
#        ▼                 ▼
# ┌───────────────────┐   ┌──────────────────────────────────────────┐
# │ NOT Exists        │   │ Exists                                   │
# │ (first ever run)  │   └────────────┬─────────────────────────────┘
# └──────────┬────────┘                │
#            │                         ▼
#            │         ┌─────────────────────────────────────────────┐
#            │         │ read_watermark()                            │
#            │         │ Filter:                                     │
#            │         │ process_type == input                       │
#            │         │ AND namespace == input                      │
#            │         └────────────┬────────────────────────────────┘
#            │                      │
#            ▼                      ▼
# ┌──────────────────────────┐     ┌────────────────────────────────────┐
# │ Use default_start_date   │     │ If matching row found:             │
# │ status = NEW             │     │ → return watermark_date + status   │
# └──────────┬───────────────┘     └────────────┬───────────────────────┘
#            │                                  │
#            ▼                                  ▼
# ┌────────────────────────────┐     ┌─────────────────────────────┐
# │ Scan folders YYYY/MM/DD    │     │ Scan folders YYYY/MM/DD     │
# │ starting after default date│     │ starting after watermark    │
# └──────────┬─────────────────┘     └────────────┬────────────────┘
#            │                                    │
#            ▼                                    ▼
# ┌──────────────────────────────┐   ┌──────────────────────────────┐
# │ get_latest_valid_path()      │   │ get_latest_valid_path()      │
# │ finds newest folder > date   │   │ finds newest folder > date   │
# └──────────┬───────────────────┘   └────────────┬─────────────────┘
#            │                                    │
#            ▼                                    ▼
# ┌──────────────────────────────┐   ┌──────────────────────────────┐
# │ No folder found              │   │ Folder found                 │
# │ → Exit pipeline safely       │   │ → Process that folder        │
# └──────────┬───────────────────┘   └────────────┬─────────────────┘
#            │                                    │
#            ▼                                    ▼
#          ┌──────────┐            ┌────────────────────────────────┐
#          │   End    │            │ Compute new watermark_date     │
#          └──────────┘            │ year/month/day → date()        │
#                                  └─────────────┬──────────────────┘
#                                                │
#                                                ▼
#                                  ┌────────────────────────────────┐
#                                  │ Call write_watermark()         │
#                                  │                                │
#                                  │ Keys used for match:           │
#                                  │ (process_type + namespace)     │
#                                  │                                │
#                                  │ Rules:                         │
#                                  │ - If date_status_update_flag   │
#                                  │     → update date + status     │
#                                  │ - If status_update_flag        │
#                                  │     → update status only       │
#                                  └────────────┬───────────────────┘
#                                               │
#                                               ▼
#                                  ┌────────────────────────────────────────────┐
#                                  │ write_watermark internal logic:            │
#                                  │                                            │
#                                  │ 1. If Delta table missing:                 │
#                                  │    → INSERT new row                        │
#                                  │                                            │
#                                  │ 2. If row (process_type + namespace) exists│
#                                  │    → UPDATE only that row                  │
#                                  │                                            │
#                                  │ 3. If row does NOT exist:                  │
#                                  │    → INSERT new row                        │
#                                  └────────────┬───────────────────────────────┘
#                                               │
#                                               ▼
#                                  ┌────────────────────────────────────────────┐
#                                  │ Safe Delta MERGE                           │
#                                  │ No overwrite                               │
#                                  │ No delete                                  │
#                                  │ No impact on other rows                    │
#                                  └────────────┬───────────────────────────────┘
#                                               ▼
#                                          ┌──────────┐
#                                          │   End    │
#                                          └──────────┘


In [ ]:
from datetime import date
from pyspark.sql.functions import col, when, lit
from pyspark.sql.types import StructType, StructField, DateType, StringType
import os
import uuid
from pyspark.sql.functions import col

In [ ]:
# Initial
param_date_status_update_flag =True
param_process_type ="Inventory"
param_status_update_flag = True
param_status = "Started"


# Once Completed
# param_date_status_update_flag =False
# param_process_type ="Inventory"
# param_status_update_flag = True
# param_status = "Completed"


## Once failed
# param_date_status_update_flag =False
# param_process_type ="Inventory"
# param_status_update_flag = True
# param_status = "Failed"


In [ ]:
# def list_dirs(path):
#     return [f.name.strip("/") for f in mssparkutils.fs.ls(path) if f.isDir]

def list_dirs(path):
    if not path:
        raise ValueError("Path cannot be empty")

    if not mssparkutils.fs.exists(path):
        raise FileNotFoundError(f"Path does not exist: {path}")
    return [
        f.name.rstrip("/") 
        for f in mssparkutils.fs.ls(path)
        if f.isDir
    ]

In [ ]:
#Must provide name-spaces else will pick directly from the directory:
param_namespaces = ["DICOM-HDS"]

workspace_name = "Imaging_dicom_checkpoint_2026_02_06"
bronze_lakehouse_name = "healthcare1_msft_bronze"


base_path = (
    f"abfss://{workspace_name}@msit-onelake.dfs.fabric.microsoft.com/{bronze_lakehouse_name}.Lakehouse/Files/Inventory/Imaging/DICOM/"
)

def get_all_name_spaces_from_dir(path):
    return list_dirs(path)

param_namespaces = param_namespaces if param_namespaces else get_all_name_spaces_from_dir(base_path)
# print(param_namespaces)

default_start_date = date(2024, 1, 1)


watermark_table_name = "CustomDicomWatermark"
watermark_date_column = "last_processed_date"
watermark_path = f"abfss://{workspace_name}@msit-onelake.dfs.fabric.microsoft.com/{bronze_lakehouse_name}.Lakehouse/Tables/{watermark_table_name}"


In [ ]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

def read_watermark(
    watermark_path,
    watermark_date_column,
    default_start_date,
    param_process_type,
    param_namespace
):
    try:
        # Check if Delta table exists
        if not DeltaTable.isDeltaTable(spark, watermark_path):
            return default_start_date, "NEW"

        # Read Delta table
        df = spark.read.format("delta").load(watermark_path)

        # Filter by composite key (process_type + namespace)
        df_filtered = df.filter(
            (col("process_type") == param_process_type) &
            (col("namespace") == param_namespace)
        )

        # If row exists, return it
        if not df_filtered.rdd.isEmpty():
            row = df_filtered.select(watermark_date_column, "status").limit(1).collect()[0]

            wm_date = row[watermark_date_column]
            status = row["status"]

            if wm_date is None:
                return default_start_date, status

            return wm_date, status

        # No matching row → treat as first run for this key
        return default_start_date, "NEW"

    except Exception as e:
        print("Error reading watermark:", e)
        return default_start_date, "ERROR"

In [ ]:
def get_latest_valid_path(base_path, watermark_dt):
    years = sorted(list_dirs(base_path), reverse=True)
    print(years)
    for y in years:
        year_path = f"{base_path}/{y}"
        months = sorted(list_dirs(year_path), reverse=True)
        for m in months:
            month_path = f"{year_path}/{m}"
            days = sorted(list_dirs(month_path), reverse=True)
            for d in days:
                folder_date = date(int(y), int(m), int(d))
                if folder_date > watermark_dt:
                    return y, m, d, f"{month_path}/{d}"
    return None

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, when, lit
from datetime import date


# --------------------------------------------------
# Validators
# --------------------------------------------------
def validate_inputs(param_process_type, param_namespace, new_watermark_date,
                    param_date_status_update_flag, param_status_update_flag):

    if not param_date_status_update_flag and not param_status_update_flag:
        raise ValueError("Both flags cannot be False")

    allowed_types = {"Inventory", "Operations"}
    if param_process_type not in allowed_types:
        raise ValueError(f"Invalid process_type: {param_process_type}")

    if not param_namespace:
        raise ValueError("namespace cannot be null or empty")

    if param_date_status_update_flag and new_watermark_date is not None:
        if not isinstance(new_watermark_date, date):
            raise ValueError("new_watermark_date must be datetime.date")


# --------------------------------------------------
# Delta reader
# --------------------------------------------------
def read_watermark_table(spark, watermark_path):
    try:
        if DeltaTable.isDeltaTable(spark, watermark_path):
            return spark.read.format("delta").load(watermark_path)
    except:
        pass
    return None


# --------------------------------------------------
# Row builder
# --------------------------------------------------
def build_watermark_df(spark, process_type, namespace, watermark_date_column, dt, status):
    return spark.createDataFrame(
        [(process_type, dt, status, namespace)],
        ["process_type", watermark_date_column, "status", "namespace"]
    )


# --------------------------------------------------
# Main function
# --------------------------------------------------

def write_watermark(
    watermark_path,
    watermark_date_column,
    new_watermark_date=None,
    param_process_type=None,
    param_namespace=None,
    param_status=None,
    param_date_status_update_flag=False,
    param_status_update_flag=False
):

    validate_inputs(
        param_process_type,
        param_namespace,
        new_watermark_date,
        param_date_status_update_flag,
        param_status_update_flag
    )

    df_existing = read_watermark_table(spark, watermark_path)

    final_date = new_watermark_date or date(1970, 1, 1)
    final_status = param_status or "NEW"

    # --------------------------------------------------
    # Case 1: No table yet → initialize
    # --------------------------------------------------
    if df_existing is None or df_existing.rdd.isEmpty():

        df_init = build_watermark_df(
            spark,
            param_process_type,
            param_namespace,
            watermark_date_column,
            final_date,
            final_status
        )

        df_init.write.format("delta").mode("overwrite").save(watermark_path)
        print(f"✅ Watermark initialized for {param_process_type} / {param_namespace}")
        return

    # --------------------------------------------------
    # Case 2: Check if row exists for (process_type, namespace)
    # --------------------------------------------------
    df_target = df_existing.filter(
        (col("process_type") == param_process_type) &
        (col("namespace") == param_namespace)
    )

    # --------------------------------------------------
    # If not exists → append via merge insert
    # --------------------------------------------------
    if df_target.rdd.isEmpty():

        df_source = build_watermark_df(
            spark,
            param_process_type,
            param_namespace,
            watermark_date_column,
            final_date,
            final_status
        )

    else:
        # --------------------------------------------------
        # Exists → update only matching row
        # --------------------------------------------------
        df_source = df_target

        if param_date_status_update_flag:
            df_source = df_source.withColumn(
                watermark_date_column, lit(new_watermark_date)
            )

        if param_status_update_flag and param_status is not None:
            df_source = df_source.withColumn("status", lit(param_status))

    # --------------------------------------------------
    # SAFE DELTA MERGE (no overwrite, no deletion)
    # --------------------------------------------------
    target = DeltaTable.forPath(spark, watermark_path)

    (
        target.alias("target")
        .merge(
            df_source.alias("source"),
            """
            target.process_type = source.process_type
            AND target.namespace = source.namespace
            """
        )
        .whenMatchedUpdate(set={
            watermark_date_column: f"source.{watermark_date_column}",
            "status": "source.status"
        })
        .whenNotMatchedInsert(values={
            "process_type": "source.process_type",
            watermark_date_column: f"source.{watermark_date_column}",
            "status": "source.status",
            "namespace": "source.namespace"
        })
        .execute()
    )

    print(f"✅ Watermark updated safely for {param_process_type} / {param_namespace}")

In [ ]:

def run_pipeline(param_date_status_update_flag, param_process_type, param_status,param_status_update_flag, param_namespaces, watermark_path, watermark_date_column, default_start_date, base_path):
    # param_namespaces is a list
    for _param_ns_ in param_namespaces:

        print("param_namespace: "+ _param_ns_)

        dir_name = _param_ns_+"-inventory"
        print("dir_name: "+ dir_name)

        inventory_file_path = os.path.join(base_path + f"{_param_ns_}/InventoryFiles/{dir_name}")
        print("inventory_file_path: "+ inventory_file_path)
        
        new_watermark_date = None
        # break

        if param_date_status_update_flag:
            # UPDATED: read_watermark must also filter by process_type + namespace
            watermark_dt, previous_status = read_watermark(
                watermark_path,
                watermark_date_column,
                default_start_date,
                param_process_type,
                _param_ns_
            )

            print("Watermark:", watermark_dt)
            print("Previous Status:", previous_status)
            print("Name Space: ", _param_ns_)
            print("Base Path: ", base_path)
            print("Inventory Path", inventory_file_path)

            result = get_latest_valid_path(inventory_file_path, watermark_dt)
            print("result:: ", result)

            if not result:
                print("No new data available")
                return

            year, month, day, selected_path = result

            print("Processing Path:", selected_path)

            new_watermark_date = date(int(year), int(month), int(day))
            print("new_watermark_date: ", new_watermark_date)

        # ---------------------------------------------------
        # Always update watermark safely (date and/or status)
        # ---------------------------------------------------
        
        write_watermark(
            watermark_path=watermark_path,
            watermark_date_column=watermark_date_column,
            new_watermark_date=new_watermark_date,
            param_process_type=param_process_type,
            param_namespace=_param_ns_,
            param_status=param_status,
            param_date_status_update_flag=param_date_status_update_flag,
            param_status_update_flag=param_status_update_flag
        )

In [ ]:
run_pipeline(param_date_status_update_flag, param_process_type, param_status,param_status_update_flag, param_namespaces, watermark_path, watermark_date_column, default_start_date, base_path)

In [1]:
import os

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
workspace_name = "Imaging_dicom_checkpoint_2026_02_06"
# watermark_table_name = "custom_dicom_watermark"
namespace = "DICOM-HDS"
bronze_lakehouse_name = "healthcare1_msft_bronze"
 
# abfss://my_ws_test_05jan2026@msit-onelake.dfs.fabric.microsoft.com/healthcare1_msft_bronze.Lakehouse/Files/External

watermark_path = f"abfss://{workspace_name}@msit-onelake.dfs.fabric.microsoft.com/{bronze_lakehouse_name}.Lakehouse/Tables/CustomDicomWatermark"
df_watermark = spark.read.format("delta").load(watermark_path)
display(df_watermark)

StatementMeta(, eb1ea41e-58cb-4392-8696-2db3a4f813bd, 5, Finished, Available, Finished, False)

AnalysisException: [PATH_NOT_FOUND] Path does not exist: abfss://Imaging_dicom_checkpoint_2026_02_06@msit-onelake.dfs.fabric.microsoft.com/healthcare1_msft_bronze.Lakehouse/Tables/CustomDicomWatermark.